# Data, features, and feature selection

This notebook covers the first half of the pipeline: where the data comes
from, how it is turned into a leakage-free feature matrix, how the market
regime is detected, and which feature groups earn their cost. The second
notebook covers the model comparison and the Pareto result.


## Data sources

The target is the Nord Pool day-ahead electricity price. Five sources feed the
feature matrix:

- ENTSO-E transparency data for cross-border scheduled exchange, net position,
  neighbouring prices, and hydro reservoir levels.
- Open-Meteo for weather (temperature, wind, solar, precipitation).
- ECB for the EUR/SEK exchange rate.
- EEX for the EUA carbon price.

Everything is joined at an ``assemble_data`` seam. That seam is also the
single network entry point, and it is backed by a Parquet cache keyed on
``(source, start, end, params)``. A re-run with the same window reads the
cache and pays no network cost.


In [ ]:
from forecast_pipeline.snapshot import SNAPSHOT_DIR, load_matrices, load_results

matrices = load_matrices(SNAPSHOT_DIR)
tables = load_results(SNAPSHOT_DIR)
print("matrix keys:", list(matrices))


## The as-of timing rule

Every exogenous feature is aligned to the forecast origin so it only uses
information available at that moment. Day-ahead values (the system load and
wind forecasts) are the one exception: they are known only after the price is
set, so group 2 (system fundamentals) is excluded from every full-features arm
to avoid train/serve skew. This rule is applied uniformly across all seven
groups.


## The seven feature groups

The full-features matrix partitions into seven groups. Group 1
(autoregressive lags, rolling statistics, calendar features, and the regime
label) is the always-present base. Groups 3 through 7 are the exogenous
blocks: cross-border, weather, hydro, commodities, and foreign exchange.
Group 2 is the excluded day-ahead block.


In [ ]:
from forecast_pipeline.feature_groups import GROUP_ORDER, EXCLUDED_GROUP

print("group order:", GROUP_ORDER)
print("excluded group:", EXCLUDED_GROUP)


Three engineering choices matter for leakage and coverage. Boundary
masking zeroes lags and rolling windows that cross a regime switch, so a
feature never mixes two frequencies. Calendar features are cyclical (sine and
cosine of hour and day of year), which keeps hour 23 and hour 0 adjacent.
Missing exogenous values are forward-filled at the source granularity.


## Regime detection

The market moved from a 60-minute to a 15-minute settlement on 2025-10-01,
and an earlier regime change was detected on 2024-11-04. A hidden-Markov
model over the price series detects these boundaries, and each fold is
labelled with the regime it belongs to. The label is a string (``regime_N``);
the darts seam re-encodes it to a trailing integer for the models that need a
numeric categorical.


In [ ]:
from forecast_pipeline.pipeline import MTU_15MIN_SWITCH_DATE

print("15-minute switch date:", MTU_15MIN_SWITCH_DATE)


Because ``assemble_data`` resolves the market time unit from the end
date, the two regimes are assembled as separate windows. The hourly window
ends the day before the switch; the 15-minute window starts on the switch
date. This keeps each window single-frequency, which the backtest requires.


## Feature selection

Feature selection runs on LightGBM, the workhorse. Three procedures rank the
seven groups:

- Forward selection adds the group that most reduces CRPS and stops when no
  group helps.
- Leave-one-group-out drops each group from the full set and records the CRPS
  loss.
- Permutation importance and mean absolute SHAP rank individual features.

The efficient set is the smallest group set whose CRPS is within 1% of the
full set. A transfer check then re-runs each other full-features arm on the
price-only base versus that efficient set, to confirm the set earns its cost
beyond the model that found it.


In [ ]:
import pandas as pd

marginal = tables["marginal_value"]
importance = tables["feature_importance"]
transfer = tables["transfer_table"]

print("marginal value (forward selection path):")
print(marginal.to_string(index=False) if not marginal.empty else "(empty)")
print()
print("top features by permutation importance:")
print(importance.head(10).to_string(index=False) if not importance.empty else "(empty)")


In [ ]:
print("efficient-set transfer (delta = efficient - price-only):")
print(transfer.to_string(index=False) if not transfer.empty else "(empty)")


## Conclusion

The forward path shows which groups reduce CRPS in the order they earn their
cost, and the transfer table shows whether that efficient set generalises
across families. Those results feed the model comparison in the second
notebook.
